In [2]:
import os
import glob
import random
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split
import keras

print("Library loaded successfully")
print("*" * 100)

# ─── CONFIGURATION GLOBAL VARIABLES ───
IMG_SIZE = 256  # Standardized input resolution for both phases
BATCH_SIZE = 4  # Batch size
EPOCHS_PHASE1 = 150  # Semantic Segmentation Focus
DATA_PATH = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG'
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/Kvasir-SEG/2_7_kv.keras"

# Switch to 'float32' to ensure 100% numerical stability with custom deep supervision losses
mixed_precision.set_global_policy('float32')


# ─── GEOMETRIC & RADIOMETRIC AUGMENTATION ───
def augment(img, mask):
    """Applies shape-preserving geometric and radiometric transformations."""
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask


# ─── PREPROCESSING & DATA INGESTION WRAPPERS ───
def process_path_segmentation(img_path, mask_path):
    """
    Advanced Preprocessing Wrapper for Kvasir-SEG Dataset.
    Applies CLAHE, Median Blur, and Mild Sharpening natively inside tf.data.
    """
    def _read_and_process(img_p, mask_p):
        img_p_str = img_p.numpy().decode('utf-8') if hasattr(img_p, 'numpy') else img_p.decode('utf-8')
        mask_p_str = mask_p.numpy().decode('utf-8') if hasattr(mask_p, 'numpy') else mask_p.decode('utf-8')
        
        # Load and convert image to RGB
        img = cv2.imread(img_p_str)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        # Load mask in grayscale
        mask = cv2.imread(mask_p_str, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        
        # STEP 1: CONTRAST ENHANCEMENT (CLAHE)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l_channel)
        
        limg = cv2.merge((cl, a_channel, b_channel))
        enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        
        # STEP 2: MEDIAN BLUR
        median_blur = cv2.medianBlur(enhanced_img, 3) 
        
        # STEP 3: MILD SHARPENING
        sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
        
        # STEP 4: NORMALIZATION
        normalized_img = sharpened / 255.0
        normalized_mask = mask / 255.0
        normalized_mask = np.where(normalized_mask > 0.5, 1.0, 0.0)
        normalized_mask = np.expand_dims(normalized_mask, axis=-1)
        
        # STEP 5: DUMMY CLASSIFICATION LABEL FOR KVASIR-SEG (All Positive)
        clf_label = np.array([1.0], dtype=np.float32)
        
        return (normalized_img.astype(np.float32), 
                normalized_mask.astype(np.float32), 
                clf_label)

    # Wrap the python function into TensorFlow graph
    [img_tensor, mask_tensor, clf_tensor] = tf.py_function(
        _read_and_process, 
        [img_path, mask_path], 
        [tf.float32, tf.float32, tf.float32]
    )
    
    # Set explicit static shapes for the flat tensors
    img_tensor.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask_tensor.set_shape([IMG_SIZE, IMG_SIZE, 1])
    clf_tensor.set_shape([1])
    
    return img_tensor, mask_tensor, clf_tensor


def load_segmentation_paths(base_path):
    all_images, all_masks = [], []
    
    img_dir = os.path.join(base_path, "images")
    mask_dir = os.path.join(base_path, "masks")
    
    if os.path.exists(img_dir) and os.path.exists(mask_dir):
        img_paths = []
        for ext in ("*.jpg", "*.JPG", "*.png", "*.PNG", "*.jpeg", "*.JPEG"):
            img_paths.extend(glob.glob(os.path.join(img_dir, ext)))
            
        for img_p in img_paths:
            filename = os.path.basename(img_p)
            mask_p = os.path.join(mask_dir, filename)
            
            if os.path.exists(mask_p):
                all_images.append(img_p)
                all_masks.append(mask_p)
            else:
                print(f"Warning: Mask missing for image: {filename}")
    else:
        print(f"Error: Target directories 'images' or 'masks' not found in root path: {base_path}")
        
    print(f"Total verified paired samples loaded: {len(all_images)}")
    return all_images, all_masks


# ─── CUSTOM MODEL ARCHITECTURE LAYERS ───
def se_block(input_tensor, ratio=16):
    """Channel-wise Squeeze-and-Excitation Attention Layer."""
    channels = input_tensor.shape[-1]
    squeeze = layers.GlobalAveragePooling2D()(input_tensor)
    excitation = layers.Dense(channels // ratio, activation='relu', use_bias=False)(squeeze)
    excitation = layers.Dense(channels, activation='sigmoid', use_bias=False)(excitation)
    excitation = layers.Reshape((1, 1, channels))(excitation)
    return layers.multiply([input_tensor, excitation])


def depthwise_seperable_conv_block(x, filters, dropout_rate=0.0, name_prefix=""):
    """Multi-Scale Parallel Kernels (3x3, 5x5, 7x7, 1x1) with Depthwise Separable Convolutions."""
    f4 = filters // 4
    b3 = layers.SeparableConv2D(f4, 3, padding='same', use_bias=False, name=f"{name_prefix}_3x3")(x)
    b3 = layers.BatchNormalization()(b3); b3 = layers.Activation('relu')(b3)
    
    b5 = layers.SeparableConv2D(f4, 5, padding='same', use_bias=False, name=f"{name_prefix}_5x5")(x)
    b5 = layers.BatchNormalization()(b5); b5 = layers.Activation('relu')(b5)
    
    b7 = layers.SeparableConv2D(f4, 7, padding='same', use_bias=False, name=f"{name_prefix}_7x7")(x)
    b7 = layers.BatchNormalization()(b7); b7 = layers.Activation('relu')(b7)
    
    b1 = layers.SeparableConv2D(f4, 1, padding='same', use_bias=False, name=f"{name_prefix}_1x1")(x)
    b1 = layers.BatchNormalization()(b1); b1 = layers.Activation('relu')(b1)
    
    merged = layers.Concatenate(name=f"{name_prefix}_concat")([b3, b5, b7, b1])
    x_se = se_block(merged, ratio=16) 
    
    if x.shape[-1] == filters:
        x_se = layers.add([x, x_se], name=f"{name_prefix}_residual")
    if dropout_rate > 0:
        x_se = layers.SpatialDropout2D(dropout_rate, name=f"{name_prefix}_dropout")(x_se)
    return x_se


@tf.keras.utils.register_keras_serializable()
class KerasPCCLayer(layers.Layer):
    """Natively computes spatial Pearson Correlation Coefficients within the network graph."""
    def __init__(self, **kwargs):
        super(KerasPCCLayer, self).__init__(**kwargs)

    def call(self, inputs):
        epsilon = 1e-8
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        std = tf.math.reduce_std(inputs, axis=-1, keepdims=True) + epsilon
        norm_inputs = (inputs - mean) / std
        pcc_matrix = tf.matmul(tf.expand_dims(norm_inputs, -1), tf.expand_dims(norm_inputs, 1))
        return layers.Flatten()(pcc_matrix)


def build_unified_segmentation_based_model(input_shape=(256, 256, 3)):
    """Constructs the dual-headed multi-task Y-Net architecture with Deep Supervision."""
    inputs = Input(shape=input_shape, name="shared_input_image")

    # ── Shared Custom Multi-Scale Encoder ──
    c1 = depthwise_seperable_conv_block(inputs, 32, name_prefix="enc_c1")
    p1 = layers.MaxPooling2D(2, name="enc_p1")(c1)

    c2 = depthwise_seperable_conv_block(p1, 64, name_prefix="enc_c2")
    p2 = layers.MaxPooling2D(2, name="enc_p2")(c2)

    c3 = depthwise_seperable_conv_block(p2, 128, dropout_rate=0.1, name_prefix="enc_c3")
    p3 = layers.MaxPooling2D(2, name="enc_p3")(c3)

    c4 = depthwise_seperable_conv_block(p3, 256, dropout_rate=0.2, name_prefix="enc_c4")
    p4 = layers.MaxPooling2D(2, name="enc_p4")(c4)
    
    # ── Shared Representation Latent Bottleneck ──
    b = depthwise_seperable_conv_block(p4, 512, dropout_rate=0.3, name_prefix="shared_bottleneck")

    # ── Output Branch A: Semantic Segmentation Decoder with Deep Supervision ──
    u4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same', name="dec_u4")(b)
    u4 = layers.concatenate([u4, c4], name="dec_concat_4")
    c5 = depthwise_seperable_conv_block(u4, 256, dropout_rate=0.2, name_prefix="dec_c5")

    ds1 = layers.Conv2D(1, 1, padding='same', name='ds1_conv')(c5)
    ds1 = layers.UpSampling2D(size=(8, 8), interpolation='bilinear', dtype='float32', name='ds1_output')(ds1)

    u3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same', name="dec_u3")(c5)
    u3 = layers.concatenate([u3, c3], name="dec_concat_3")
    c6 = depthwise_seperable_conv_block(u3, 128, dropout_rate=0.1, name_prefix="dec_c6")

    ds2 = layers.Conv2D(1, 1, padding='same', name='ds2_conv')(c6)
    ds2 = layers.UpSampling2D(size=(4, 4), interpolation='bilinear', dtype='float32', name='ds2_output')(ds2)

    u2 = layers.Conv2DTranspose(64, 2, strides=2, padding='same', name="dec_u2")(c6)
    u2 = layers.concatenate([u2, c2], name="dec_concat_2")
    c7 = depthwise_seperable_conv_block(u2, 64, name_prefix="dec_c7")

    ds3 = layers.Conv2D(1, 1, padding='same', name='ds3_conv')(c7)
    ds3 = layers.UpSampling2D(size=(2, 2), interpolation='bilinear', dtype='float32', name='ds3_output')(ds3)

    u1 = layers.Conv2DTranspose(32, 2, strides=2, padding='same', name="dec_u1")(c7)
    u1 = layers.concatenate([u1, c1], name="dec_concat_1")
    c8 = depthwise_seperable_conv_block(u1, 32, name_prefix="dec_c8")

    segmentation_output = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='seg_output')(c8)

    # ── Output Branch B: Segmentation-Guided Classification Head ──
    pd_cnn = layers.SeparableConv2D(32, (3, 3), padding='same', activation='relu', name="clf_pd_cnn")(b)
    p_pool = layers.MaxPooling2D((2, 2), name="clf_maxpool")(pd_cnn)
    flat_features = layers.GlobalAveragePooling2D(name="clf_gap")(p_pool)
    pcc_features = KerasPCCLayer(name="clf_pcc_layer")(flat_features)
    standardized = layers.BatchNormalization(name="clf_zscore_bn")(pcc_features)
    
    d1 = layers.Dense(64, activation='relu', name="clf_dense1")(standardized)
    d1 = layers.Dropout(0.3, name="clf_dropout")(d1)
    classification_output = layers.Dense(1, activation='sigmoid', dtype='float32', name='clf_output')(d1)

    return models.Model(
        inputs=inputs, 
        outputs=[segmentation_output, ds1, ds2, ds3, classification_output], 
        name="Thesis_DeepSupervised_YNet"
    )


# ─── EVALUATION METRICS & LOSS FUNCTIONS WITH STABILITY CLIPPING ───
@keras.saving.register_keras_serializable()
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    # Clip predictions to prevent 0 or 1 edge cases disrupting division
    y_pred_f = tf.clip_by_value(tf.cast(tf.reshape(y_pred, [-1]), tf.float32), 1e-7, 1.0 - 1e-7)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

@keras.saving.register_keras_serializable()
def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.clip_by_value(tf.cast(tf.reshape(y_pred, [-1]), tf.float32), 1e-7, 1.0 - 1e-7)
    
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    fp = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    
    smooth = 1e-6
    tversky_index = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
    
    # Clip the inner pow operator components to explicitly secure against nan gradients
    loss_val = tf.math.pow(tf.clip_by_value(1.0 - tversky_index, 1e-7, 1.0), gamma)
    return loss_val

@keras.saving.register_keras_serializable()
def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)

@keras.saving.register_keras_serializable()
def tnr_metric(y_true, y_pred):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.clip_by_value(tf.cast(tf.reshape(y_pred, [-1]), tf.float32), 1e-7, 1.0 - 1e-7)
    
    true_negatives = tf.reduce_sum(tf.cast((1.0 - y_true_f) * (1.0 - y_pred_f), tf.float32))
    false_positives = tf.reduce_sum(tf.cast((1.0 - y_true_f) * y_pred_f, tf.float32))
    return (true_negatives + 1e-6) / (true_negatives + false_positives + 1e-6)

@keras.saving.register_keras_serializable()
def f2_segmentation(y_true, y_pred):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    y_pred_f = tf.cast(y_pred_f > 0.5, tf.float32)
    
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fp = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    return (5.0 * tp + 1e-6) / (5.0 * tp + 4.0 * fn + fp + 1e-6)


# =====================================================================
# STEP 1: PARSE PIPELINE DIRECTORIES & RUN COMPONENT WRAPPERS
# =====================================================================

# Initialize allocation strategy for physical compute hardware
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus: 
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ Tensor Graph Allocation Locked over available GPU nodes.")
    except RuntimeError as e: 
        print(e)

print("\n[Data Ingestion] Parsing Kvasir-SEG Localization Data...")
seg_imgs, seg_masks = load_segmentation_paths(DATA_PATH)
s_train_x, s_temp_x, s_train_y, s_temp_y = train_test_split(seg_imgs, seg_masks, test_size=0.2, random_state=42)
s_val_x, s_test_x, s_val_y, s_test_y = train_test_split(s_temp_x, s_temp_y, test_size=0.5, random_state=42)

# =====================================================================
# STEP 2: GENERATE HIGH-THROUGHPUT ITERATORS WITH *ARGS UNPACKING
# =====================================================================
print("\n⚡ Initializing Multi-Threaded High-Throughput Iterators via AUTOTUNE...")

def apply_tuple_augmentation(*args):
    """Safely unpacks the graph stream and applies augmentation strictly to the image."""
    if len(args) == 1:
        args = args[0]
    image, mask, clf_label = args
    augmented_image, augmented_mask = augment(image, mask)
    return augmented_image, augmented_mask, clf_label

def map_tuple_to_model_outputs(*args):
    """Restructures flat elements into the required 5-output structure for compiling."""
    if len(args) == 1:
        args = args[0]
    image, mask, clf_label = args
    return image, {
        'seg_output': mask,
        'ds1_output': mask,
        'ds2_output': mask,
        'ds3_output': mask,
        'clf_output': clf_label
    }

# --- Pipeline Instantiation ---
train_seg_ds = tf.data.Dataset.from_tensor_slices((s_train_x, s_train_y)) \
    .shuffle(len(s_train_x)) \
    .map(process_path_segmentation, num_parallel_calls=tf.data.AUTOTUNE) \
    .map(apply_tuple_augmentation, num_parallel_calls=tf.data.AUTOTUNE) \
    .map(map_tuple_to_model_outputs, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

val_seg_ds = tf.data.Dataset.from_tensor_slices((s_val_x, s_val_y)) \
    .map(process_path_segmentation, num_parallel_calls=tf.data.AUTOTUNE) \
    .map(map_tuple_to_model_outputs, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

test_seg_ds = tf.data.Dataset.from_tensor_slices((s_test_x, s_test_y)) \
    .map(process_path_segmentation, num_parallel_calls=tf.data.AUTOTUNE) \
    .map(map_tuple_to_model_outputs, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

# =====================================================================
# STEP 3: GRAPH INSTANTIATION & MULTI-HEAD COMPILATION
# =====================================================================
print("\n🏗️ Compiling Dual-Head Computation Topology...")
model = build_unified_segmentation_based_model()

print("\n=======================================================")
print("[PHASE 1] Optimizing Localization Ensembles & Structural Layers")
print("=======================================================")

# Freeze classification branch entirely to protect gradients during Phase 1
for layer in model.layers:
    if "clf_" in layer.name:
        layer.trainable = False
    else:
        layer.trainable = True

# Compile utilizing Adam with a slightly safer learning rate (5e-4) to avoid explosion
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss={
        'seg_output': focal_tversky_loss,
        'ds1_output': focal_tversky_loss,
        'ds2_output': focal_tversky_loss,
        'ds3_output': focal_tversky_loss,
        'clf_output': None
    },
    loss_weights={
        'seg_output': 1.0,
        'ds1_output': 0.3,
        'ds2_output': 0.3,
        'ds3_output': 0.3,
        'clf_output': 0.0
    },
    metrics={
        'seg_output': [
            'accuracy',                                       
            tf.keras.metrics.Recall(name='recall'),          
            tnr_metric,
            dice_coef,
            iou_metric,
            f2_segmentation
        ]
    }
)


# =====================================================================
# STEP 4: TRAINING EXECUTION
# =====================================================================
phase1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH, 
        monitor='val_seg_output_dice_coef', 
        save_best_only=True, 
        mode='max', 
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=6, 
        min_lr=1e-4, 
        verbose=1
    )
]

# Run Phase 1 Model Fit
history_p1 = model.fit(
    train_seg_ds, 
    validation_data=val_seg_ds, 
    epochs=EPOCHS_PHASE1, 
    callbacks=phase1_callbacks, 
    verbose=1
)

print("[PHASE SEGMENTATION-TRAINING COMPLETE] Spatial boundaries locked successfully.")

Library loaded successfully
****************************************************************************************************
✓ Tensor Graph Allocation Locked over available GPU nodes.

[Data Ingestion] Parsing Kvasir-SEG Localization Data...
Total verified paired samples loaded: 1000

⚡ Initializing Multi-Threaded High-Throughput Iterators via AUTOTUNE...


I0000 00:00:1782965425.671452   14138 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9



🏗️ Compiling Dual-Head Computation Topology...

[PHASE 1] Optimizing Localization Ensembles & Structural Layers
Epoch 1/150


I0000 00:00:1782965460.473043   17626 service.cc:153] XLA service 0x72e388120570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782965460.473117   17626 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.1)
I0000 00:00:1782965461.782026   17626 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782965466.920695   17626 cuda_dnn.cc:461] Loaded cuDNN version 92101
I0000 00:00:1782965468.258755   17626 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_37334__.337
E0000 00:00:1782965473.248681   17626 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1782965488.751040   17626 cuda_timer.cc:87] Delay kernel time

200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - ds1_output_loss: 0.6039 - ds2_output_loss: 0.6289 - ds3_output_loss: 0.6615 - loss: 1.2457 - seg_output_accuracy: 0.6036 - seg_output_dice_coef: 0.3161 - seg_output_f2_segmentation: 0.5549 - seg_output_iou_metric: 0.2430 - seg_output_loss: 0.6774 - seg_output_recall: 0.8525 - seg_output_tnr_metric: 0.5213
Epoch 1: val_seg_output_dice_coef improved from None to 0.23787, saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/Kvasir-SEG/2_7_kv.keras

Epoch 1: finished saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/Kvasir-SEG/2_7_kv.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 158s 169ms/step - ds1_output_loss: 0.5693 - ds2_output_loss: 0.5750 - ds3_output_loss: 0.5852 - loss: 1.1511 - seg_output_accuracy: 0.6540 - seg_output_dice_coef: 0.3615 - seg_output_f2_segmentation: 0.5884 - seg_output_iou_metric: 0.2751 - seg_output_loss: 0.6323 - seg_output_recall: 0.8517 - seg_output_tnr_metri

In [3]:
# =============================================================================
# 📊 PRINT PEAK PERFORMANCE METRICS FOR THESIS REPORT
# =============================================================================
print("=====================================================================")
print("🏆 Unified Model: Peak Performance Metrics (Phase 1 & Phase 2)")
print("=====================================================================")

# 🧠 1. PHASE 1: SEMANTIC SEGMENTATION PERFORMANCE
h1 = history_p1.history


dice_key = [k for k in h1.keys() if 'dice' in k and 'val' not in k][0]
val_dice_key = [k for k in h1.keys() if 'dice' in k and 'val' in k][0]
iou_key = [k for k in h1.keys() if 'iou' in k and 'val' not in k][0]
val_iou_key = [k for k in h1.keys() if 'iou' in k and 'val' in k][0]


acc_key = [k for k in h1.keys() if ('acc' in k or 'accuracy' in k) and 'val' not in k][0]
val_acc_key = [k for k in h1.keys() if ('acc' in k or 'accuracy' in k) and 'val' in k][0]
rec_key = [k for k in h1.keys() if 'recall' in k and 'val' not in k][0]
val_rec_key = [k for k in h1.keys() if 'recall' in k and 'val' in k][0]
tnr_key = [k for k in h1.keys() if 'tnr' in k and 'val' not in k][0]
val_tnr_key = [k for k in h1.keys() if 'tnr' in k and 'val' in k][0]


best_p1_idx = np.argmax(h1[val_dice_key])

print(f"\n📍 [PHASE 1: SEGMENTATION (Best Epoch: {best_p1_idx + 1})]")
print(f"   • Best Training Dice Score    : {h1[dice_key][best_p1_idx]:.4f}")
print(f"   • Best Validation Dice Score  : {h1[val_dice_key][best_p1_idx]:.4f} 🌟")
print(f"   • Associated Training IoU     : {h1[iou_key][best_p1_idx]:.4f}")
print(f"   • Associated Validation IoU   : {h1[val_iou_key][best_p1_idx]:.4f}")

print(f"   • Associated Training Acc     : {h1[acc_key][best_p1_idx]:.4f}")
print(f"   • Associated Validation Acc   : {h1[val_acc_key][best_p1_idx]:.4f}")
print(f"   • Associated Training Recall  : {h1[rec_key][best_p1_idx]:.4f}")
print(f"   • Associated Validation Recall: {h1[val_rec_key][best_p1_idx]:.4f}")
print(f"   • Associated Training TNR     : {h1[tnr_key][best_p1_idx]:.4f}")
print(f"   • Associated Validation TNR   : {h1[val_tnr_key][best_p1_idx]:.4f}")




🏆 Unified Model: Peak Performance Metrics (Phase 1 & Phase 2)

📍 [PHASE 1: SEGMENTATION (Best Epoch: 147)]
   • Best Training Dice Score    : 0.8802
   • Best Validation Dice Score  : 0.8608 🌟
   • Associated Training IoU     : 0.7955
   • Associated Validation IoU   : 0.7630
   • Associated Training Acc     : 0.9643
   • Associated Validation Acc   : 0.9592
   • Associated Training Recall  : 0.9090
   • Associated Validation Recall: 0.8712
   • Associated Training TNR     : 0.9738
   • Associated Validation TNR   : 0.9742


In [20]:
# =====================================================================
# STEP 5: FINAL EVALUATION ON INDEPENDENT TESTING DATASET
# =====================================================================
print("\n=======================================================")
print("🔮 Evaluating Final Model Performance on Unseen Test Dataset (10%)")
print("=======================================================")

# ১. টেস্ট ডেটাসেট দিয়ে মডেলের সম্পূর্ণ মূল্যায়ন করা এবং অ্যারে নেওয়া
test_results = model.evaluate(test_seg_ds, verbose=1)
print( "test_results", test_results)
print("metrics_dict", metrics_dict)
print("\n[PHASE 1: SEGMENTATION (Independent Unseen Test Set Results)]")

# index 6 = Dice, index 8 = IoU, index 5 = Accuracy, index 9 = Recall, index 10 = TNR, index 7 = F2


test_dice = test_results[6] if len(test_results) > 6 else 0.0
print(f"   • Associated Testing Dice Score  : {test_dice:.4f} 🌟")


test_iou = test_results[8] if len(test_results) > 8 else 0.0
print(f"   • Associated Testing IoU        : {test_iou:.4f}")


test_acc = test_results[5] if len(test_results) > 5 else 0.0
print(f"   • Associated Testing Acc        : {test_acc:.4f}")


test_recall = test_results[9] if len(test_results) > 9 else 0.0
print(f"   • Associated Testing Recall     : {test_recall:.4f}")


test_tnr = test_results[10] if len(test_results) > 10 else 0.0
print(f"   • Associated Testing TNR        : {test_tnr:.4f}")


test_f2 = test_results[7] if len(test_results) > 7 else 0.0
print(f"   • Associated Testing F2 Score   : {test_f2:.4f}")


🔮 Evaluating Final Model Performance on Unseen Test Dataset (10%)
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - ds1_output_loss: 0.2594 - ds2_output_loss: 0.2413 - ds3_output_loss: 0.2360 - loss: 0.4568 - seg_output_accuracy: 0.9470 - seg_output_dice_coef: 0.8439 - seg_output_f2_segmentation: 0.8490 - seg_output_iou_metric: 0.7435 - seg_output_loss: 0.2358 - seg_output_recall: 0.8449 - seg_output_tnr_metric: 0.9671
test_results [0.456798791885376, 0.23579853773117065, 0.25942695140838623, 0.241264209151268, 0.23597656190395355, 0.9470115900039673, 0.8439465165138245, 0.8490085601806641, 0.7435351014137268, 0.8448702692985535, 0.9671385884284973]
metrics_dict {'loss': 0.456798791885376, 'compile_metrics': 0.23579853773117065, 'seg_output_loss': 0.25942695140838623, 'ds1_output_loss': 0.241264209151268, 'ds2_output_loss': 0.23597656190395355, 'ds3_output_loss': 0.9470115900039673}

[PHASE 1: SEGMENTATION (Independent Unseen Test Set Results)]
   • Associated Testing Dice Score  : 0.8439 🌟
 